# Microsoft Recommenders 评估指标 PyTorch 版

这份 notebook 对应 Microsoft Recommenders 的 `examples/03_evaluate/evaluation.ipynb`，但这里不使用 Spark，也不依赖 `recommenders` 包。我们用 Pandas 组织示例数据，用 PyTorch/Numpy 手写核心指标，方便你直接看清每个推荐系统评估指标到底在算什么。

本 notebook 覆盖两类指标：

- 评分预测指标：RMSE、MAE、R2、解释方差。
- 排序推荐指标：Precision@K、Recall@K、NDCG@K、MAP@K、AUC、LogLoss。


## 1. 导入依赖与全局字段

字段命名沿用推荐系统里最常见的三元组形式：用户、物品、真实反馈，以及模型预测分数。这里的预测分数越大，表示模型越想把该物品排在前面。

In [ ]:
import numpy as np
import pandas as pd
import torch

COL_USER = "UserId"
COL_ITEM = "MovieId"
COL_RATING = "Rating"
COL_PREDICTION = "Prediction"

torch.manual_seed(42)
np.random.seed(42)


## 2. 构造示例数据

`df_true` 表示真实反馈，`df_pred` 表示模型给出的推荐顺序或预测分数。这个小数据集来自 Microsoft Recommenders 示例的思想：不同用户有不同数量的真实交互，也有不同的推荐列表。

In [ ]:
df_true = pd.DataFrame({
    COL_USER: [1, 1, 1, 2, 2, 2, 2, 2, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3],
    COL_ITEM: [1, 2, 3, 1, 4, 5, 6, 7, 2, 5, 6, 8, 9, 10, 11, 12, 13, 14],
    COL_RATING: [5, 4, 3, 5, 5, 3, 3, 1, 5, 5, 5, 4, 4, 3, 3, 3, 2, 1],
})

df_pred = pd.DataFrame({
    COL_USER: [1, 1, 1, 2, 2, 2, 2, 2, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3],
    COL_ITEM: [3, 10, 12, 10, 3, 5, 11, 13, 4, 10, 7, 13, 1, 3, 5, 2, 11, 14],
    COL_PREDICTION: [14, 13, 12, 14, 13, 12, 11, 10, 14, 13, 12, 11, 10, 9, 8, 7, 6, 5],
})

display(df_true.head())
display(df_pred.head())


## 3. 评分预测指标

评分预测指标把推荐看成回归问题：模型需要预测用户对物品的评分。我们只在真实数据和预测数据共同出现的 `(用户, 物品)` 上计算。

In [ ]:
def align_rating_rows(df_true, df_pred):
    merged = df_true.merge(df_pred, on=[COL_USER, COL_ITEM], how="inner")
    y_true = torch.tensor(merged[COL_RATING].to_numpy(dtype=np.float32))
    y_pred = torch.tensor(merged[COL_PREDICTION].to_numpy(dtype=np.float32))
    return merged, y_true, y_pred


def rmse(y_true, y_pred):
    return torch.sqrt(torch.mean((y_true - y_pred) ** 2)).item()


def mae(y_true, y_pred):
    return torch.mean(torch.abs(y_true - y_pred)).item()


def r2_score_torch(y_true, y_pred):
    ss_res = torch.sum((y_true - y_pred) ** 2)
    ss_tot = torch.sum((y_true - torch.mean(y_true)) ** 2)
    return (1 - ss_res / ss_tot).item()


def explained_variance_torch(y_true, y_pred):
    variance_error = torch.var(y_true - y_pred, unbiased=False)
    variance_true = torch.var(y_true, unbiased=False)
    return (1 - variance_error / variance_true).item()

merged_rating, y_true, y_pred = align_rating_rows(df_true, df_pred)
print(f"共同出现的评分样本数：{len(merged_rating)}")
print(f"RMSE：{rmse(y_true, y_pred):.4f}")
print(f"MAE：{mae(y_true, y_pred):.4f}")
print(f"R2：{r2_score_torch(y_true, y_pred):.4f}")
print(f"解释方差：{explained_variance_torch(y_true, y_pred):.4f}")


## 4. 排序推荐指标

排序指标更符合 Top-K 推荐场景：模型不一定要预测准确评分，而是要把用户真正感兴趣的物品排在前面。这里我们把真实评分大于等于 4 的物品视为相关物品。

In [ ]:
def make_relevance(df_true, threshold=4):
    relevant = df_true[df_true[COL_RATING] >= threshold]
    return relevant.groupby(COL_USER)[COL_ITEM].apply(set).to_dict()


def top_k_items(df_pred, k):
    ranked = df_pred.sort_values([COL_USER, COL_PREDICTION], ascending=[True, False])
    return ranked.groupby(COL_USER)[COL_ITEM].apply(lambda x: list(x.head(k))).to_dict()


def precision_at_k(df_true, df_pred, k=3, threshold=4):
    relevant_by_user = make_relevance(df_true, threshold)
    pred_by_user = top_k_items(df_pred, k)
    scores = []
    for user, pred_items in pred_by_user.items():
        hits = len(set(pred_items) & relevant_by_user.get(user, set()))
        scores.append(hits / k)
    return float(np.mean(scores))


def recall_at_k(df_true, df_pred, k=3, threshold=4):
    relevant_by_user = make_relevance(df_true, threshold)
    pred_by_user = top_k_items(df_pred, k)
    scores = []
    for user, relevant_items in relevant_by_user.items():
        pred_items = pred_by_user.get(user, [])
        hits = len(set(pred_items) & relevant_items)
        scores.append(hits / max(len(relevant_items), 1))
    return float(np.mean(scores))


def dcg_at_k(labels, k):
    labels = np.asarray(labels, dtype=np.float32)[:k]
    discounts = 1 / np.log2(np.arange(2, labels.size + 2))
    return float(np.sum(labels * discounts))


def ndcg_at_k(df_true, df_pred, k=3, threshold=4):
    relevant_by_user = make_relevance(df_true, threshold)
    pred_by_user = top_k_items(df_pred, k)
    scores = []
    for user, pred_items in pred_by_user.items():
        labels = [1 if item in relevant_by_user.get(user, set()) else 0 for item in pred_items]
        ideal_labels = sorted(labels, reverse=True)
        ideal_dcg = dcg_at_k(ideal_labels, k)
        scores.append(dcg_at_k(labels, k) / ideal_dcg if ideal_dcg > 0 else 0.0)
    return float(np.mean(scores))


def map_at_k(df_true, df_pred, k=3, threshold=4):
    relevant_by_user = make_relevance(df_true, threshold)
    pred_by_user = top_k_items(df_pred, k)
    scores = []
    for user, pred_items in pred_by_user.items():
        relevant_items = relevant_by_user.get(user, set())
        hit_count = 0
        precision_sum = 0.0
        for rank, item in enumerate(pred_items[:k], start=1):
            if item in relevant_items:
                hit_count += 1
                precision_sum += hit_count / rank
        scores.append(precision_sum / min(len(relevant_items), k) if relevant_items else 0.0)
    return float(np.mean(scores))

print(f"Precision@3：{precision_at_k(df_true, df_pred, k=3):.4f}")
print(f"Recall@3：{recall_at_k(df_true, df_pred, k=3):.4f}")
print(f"NDCG@3：{ndcg_at_k(df_true, df_pred, k=3):.4f}")
print(f"MAP@3：{map_at_k(df_true, df_pred, k=3):.4f}")


## 5. AUC 与 LogLoss

AUC 和 LogLoss 常用于点击率或二分类推荐。这里把评分大于 3 的样本视为正样本，并把预测分数缩放到 `[0, 1]`。

In [ ]:
def minmax_scale_torch(values):
    values = torch.tensor(values, dtype=torch.float32)
    return (values - values.min()) / (values.max() - values.min())


def auc_torch(labels, scores):
    labels = torch.tensor(labels, dtype=torch.float32)
    scores = torch.tensor(scores, dtype=torch.float32)
    order = torch.argsort(scores, descending=True)
    labels = labels[order]
    positives = torch.sum(labels == 1).item()
    negatives = torch.sum(labels == 0).item()
    if positives == 0 or negatives == 0:
        return float("nan")
    true_positive = torch.cumsum(labels == 1, dim=0).float()
    false_positive = torch.cumsum(labels == 0, dim=0).float()
    tpr = torch.cat([torch.tensor([0.0]), true_positive / positives])
    fpr = torch.cat([torch.tensor([0.0]), false_positive / negatives])
    return torch.trapz(tpr, fpr).item()


def logloss_torch(labels, probabilities, eps=1e-12):
    labels = torch.tensor(labels, dtype=torch.float32)
    probabilities = torch.tensor(probabilities, dtype=torch.float32).clamp(eps, 1 - eps)
    loss = -(labels * torch.log(probabilities) + (1 - labels) * torch.log(1 - probabilities))
    return loss.mean().item()

merged_binary = df_true.merge(df_pred, on=[COL_USER, COL_ITEM], how="inner")
labels = (merged_binary[COL_RATING].to_numpy() > 3).astype(np.float32)
probabilities = minmax_scale_torch(merged_binary[COL_PREDICTION].to_numpy()).numpy()

print(f"AUC：{auc_torch(labels, probabilities):.4f}")
print(f"LogLoss：{logloss_torch(labels, probabilities):.4f}")


## 6. 怎么选择指标

- 预测评分时，优先看 RMSE、MAE、R2、解释方差。
- 做 Top-K 推荐时，优先看 Recall@K、NDCG@K、MAP@K。
- 做点击率或二分类排序时，常看 AUC 和 LogLoss。

真实业务里通常不会只看一个指标。比如新闻推荐往往同时关注 AUC、MRR、NDCG@5、NDCG@10，因为它既要判断点不点击，也要关心正样本是否排在候选列表前面。